# PCA降维示例

## 目标

理解主成分分析（PCA）的原理和应用。

- 理解PCA的数学原理
- 从零实现PCA算法
- 与Scikit-learn的对比
- 可视化降维效果

## 1. 数据准备

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris, load_wine, load_breast_cancer, make_blobs
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import pandas as pd

np.random.seed(42)

# 加载Iris数据集
iris = load_iris()
X = iris.data
y = iris.target
feature_names = iris.feature_names
target_names = iris.target_names

print(f"数据集形状: {X.shape}")
print(f"类别数量: {len(target_names)}")
print(f"类别名称: {target_names}")
print(f"特征名称: {feature_names}")

In [ ]:
# 标准化数据（PCA对尺度敏感）
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("标准化后的数据统计:")
print(f"均值: {X_scaled.mean(axis=0)}")
print(f"标准差: {X_scaled.std(axis=0)}")

## 2. PCA数学原理

In [ ]:
def pca_manual(X, n_components=2):
    """
    从零实现的PCA算法
    
    步骤：
    1. 中心化数据
    2. 计算协方差矩阵
    3. 计算特征值和特征向量
    4. 选择前k个主成分
    5. 投影数据
    """
    # 1. 中心化（假设数据已经标准化）
    X_centered = X - X.mean(axis=0)
    
    # 2. 计算协方差矩阵
    n_samples = X.shape[0]
    cov_matrix = (X_centered.T @ X_centered) / (n_samples - 1)
    
    # 3. 计算特征值和特征向量
    eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)
    
    # 4. 按特征值降序排序
    sorted_indices = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[sorted_indices]
    eigenvectors = eigenvectors[:, sorted_indices]
    
    # 5. 选择前k个主成分
    components = eigenvectors[:, :n_components]
    
    # 6. 投影数据
    X_transformed = X_centered @ components
    
    # 计算解释方差比例
    explained_variance_ratio = eigenvalues[:n_components] / eigenvalues.sum()
    
    return X_transformed, components, explained_variance_ratio, eigenvalues

print("手动PCA函数定义完成")

## 3. 训练手动实现的PCA

In [ ]:
# 使用手动PCA降维到2维
X_pca_manual, components_manual, explained_var_manual, eigenvalues_manual = pca_manual(X_scaled, n_components=2)

print("手动PCA结果:")
print(f"  降维后形状: {X_pca_manual.shape}")
print(f"  解释方差比例: {explained_var_manual}")
print(f"  累计解释方差比例: {explained_var_manual.sum():.4f}")
print(f"\n  主成分（特征向量）:")
for i, (name, comp) in enumerate(zip(['PC1', 'PC2'], components_manual.T)):
    print(f"    {name}: {comp}")

## 4. 使用Scikit-learn的PCA

In [ ]:
# 使用Scikit-learn的PCA
pca_sklearn = PCA(n_components=2)
X_pca_sklearn = pca_sklearn.fit_transform(X_scaled)

print("Scikit-learn PCA结果:")
print(f"  降维后形状: {X_pca_sklearn.shape}")
print(f"  解释方差比例: {pca_sklearn.explained_variance_ratio_}")
print(f"  累计解释方差比例: {pca_sklearn.explained_variance_ratio_.sum():.4f}")
print(f"\n  主成分（特征向量）:")
for i, (name, comp) in enumerate(zip(['PC1', 'PC2'], pca_sklearn.components_.T)):
    print(f"    {name}: {comp}")

print(f"\n与手动实现的差异:")
print(f"  数据差异: {np.abs(X_pca_manual - X_pca_sklearn).max():.10f}")
print(f"  主成分差异: {np.abs(np.abs(components_manual) - np.abs(pca_sklearn.components_)).max():.10f}")

## 5. 可视化PCA结果

In [ ]:
# 可视化对比
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 手动实现
colors = ['red', 'green', 'blue']
for i, target_name in enumerate(target_names):
    mask = (y == i)
    axes[0].scatter(X_pca_manual[mask, 0], X_pca_manual[mask, 1],
                  c=colors[i], label=target_name, alpha=0.7)
axes[0].set_xlabel(f'PC1 ({explained_var_manual[0]:.2%} 方差)')
axes[0].set_ylabel(f'PC2 ({explained_var_manual[1]:.2%} 方差)')
axes[0].set_title('手动PCA')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Scikit-learn
for i, target_name in enumerate(target_names):
    mask = (y == i)
    axes[1].scatter(X_pca_sklearn[mask, 0], X_pca_sklearn[mask, 1],
                  c=colors[i], label=target_name, alpha=0.7)
axes[1].set_xlabel(f'PC1 ({pca_sklearn.explained_variance_ratio_[0]:.2%} 方差)')
axes[1].set_ylabel(f'PC2 ({pca_sklearn.explained_variance_ratio_[1]:.2%} 方差)')
axes[1].set_title('Scikit-learn PCA')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. 特征向量的解释

In [ ]:
# 创建主成分载荷矩阵
loadings_df = pd.DataFrame(
    pca_sklearn.components_.T,
    index=feature_names,
    columns=['PC1', 'PC2']
)

print("主成分载荷矩阵（特征向量）:")
print(loadings_df.round(4))

print("\n主成分解释:")
print("PC1 主要由花萼和花瓣的长度和宽度共同决定（正相关）")
print("PC2 主要区分花萼和花瓣的特征（花萼特征为正，花瓣特征为负）")

## 7. 累计解释方差曲线

In [ ]:
# 使用所有主成分
pca_full = PCA()
pca_full.fit(X_scaled)

# 计算累计解释方差
cumulative_variance = np.cumsum(pca_full.explained_variance_ratio_)

# 可视化
plt.figure(figsize=(12, 5))

# 单个主成分的解释方差
plt.subplot(1, 2, 1)
bars = plt.bar(range(1, len(pca_full.explained_variance_ratio_) + 1),
              pca_full.explained_variance_ratio_, alpha=0.7)
plt.xlabel('主成分')
plt.ylabel('解释方差比例')
plt.title('各主成分的解释方差')
plt.grid(True, alpha=0.3, axis='y')
plt.xticks(range(1, len(pca_full.explained_variance_ratio_) + 1))

# 添加数值标签
for i, bar in enumerate(bars):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{height:.3f}', ha='center', va='bottom')

# 累计解释方差
plt.subplot(1, 2, 2)
plt.plot(range(1, len(cumulative_variance) + 1), cumulative_variance,
         'bo-', linewidth=2, markersize=8)
plt.axhline(y=0.95, color='r', linestyle='--', label='95% 阈值')
plt.axhline(y=0.90, color='g', linestyle='--', label='90% 阈值')
plt.xlabel('主成分数量')
plt.ylabel('累计解释方差比例')
plt.title('累计解释方差曲线')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xticks(range(1, len(cumulative_variance) + 1))

plt.tight_layout()
plt.show()

print("累计解释方差:")
for i, cum_var in enumerate(cumulative_variance, 1):
    print(f"  {i}个主成分: {cum_var:.4f} ({cum_var:.2%})")

## 8. 特征重要性分析

In [ ]:
# 计算每个特征对主成分的贡献
feature_contributions = pd.DataFrame(
    index=feature_names,
    columns=['PC1', 'PC2', 'PC3', 'PC4']
)

for i in range(4):
    feature_contributions.iloc[:, i] = pca_full.components_[i] ** 2

# 计算每个特征的总贡献
feature_contributions['总贡献'] = feature_contributions.sum(axis=1)
feature_contributions = feature_contributions.sort_values('总贡献', ascending=False)

print("各特征对主成分的贡献（平方）:")
print(feature_contributions.round(4))

# 可视化
plt.figure(figsize=(10, 6))
plt.barh(range(len(feature_contributions)), feature_contributions['总贡献'],
        color='steelblue', alpha=0.7)
plt.yticks(range(len(feature_contributions)), feature_contributions.index)
plt.xlabel('总贡献（平方和）')
plt.title('各特征对主成分的总贡献')
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

## 9. 不同主成分数量的对比

In [ ]:
# 测试不同主成分数量下的分类性能
n_components_list = [1, 2, 3, 4]
train_scores = []
test_scores = []
cumulative_vars = []

# 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.3, random_state=42, stratify=y
)

for n_comp in n_components_list:
    # PCA降维
    pca_temp = PCA(n_components=n_comp)
    X_train_pca = pca_temp.fit_transform(X_train)
    X_test_pca = pca_temp.transform(X_test)
    
    # 训练逻辑回归
    lr = LogisticRegression(random_state=42, max_iter=1000)
    lr.fit(X_train_pca, y_train)
    
    train_scores.append(lr.score(X_train_pca, y_train))
    test_scores.append(lr.score(X_test_pca, y_test))
    cumulative_vars.append(pca_temp.explained_variance_ratio_.sum())

# 可视化
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 分类性能
axes[0].plot(n_components_list, train_scores, 'b-o', label='训练集', linewidth=2, markersize=8)
axes[0].plot(n_components_list, test_scores, 'r-o', label='测试集', linewidth=2, markersize=8)
axes[0].set_xlabel('主成分数量')
axes[0].set_ylabel('准确率')
axes[0].set_title('分类性能 vs 主成分数量')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_xticks(n_components_list)

# 累计解释方差
axes[1].plot(n_components_list, cumulative_vars, 'g-o', linewidth=2, markersize=8)
axes[1].axhline(y=0.95, color='r', linestyle='--', label='95%')
axes[1].axhline(y=0.90, color='orange', linestyle='--', label='90%')
axes[1].set_xlabel('主成分数量')
axes[1].set_ylabel('累计解释方差比例')
axes[1].set_title('累计解释方差 vs 主成分数量')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_xticks(n_components_list)

plt.tight_layout()
plt.show()

print("不同主成分数量的结果:")
results_df = pd.DataFrame({
    '主成分数量': n_components_list,
    '累计解释方差': cumulative_vars,
    '训练集准确率': train_scores,
    '测试集准确率': test_scores
})
print(results_df.round(4).to_string(index=False))

## 10. 3D可视化

In [ ]:
# 降维到3维
pca_3d = PCA(n_components=3)
X_pca_3d = pca_3d.fit_transform(X_scaled)

# 3D散点图
fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection='3d')

colors = ['red', 'green', 'blue']
for i, target_name in enumerate(target_names):
    mask = (y == i)
    ax.scatter(X_pca_3d[mask, 0], X_pca_3d[mask, 1], X_pca_3d[mask, 2],
              c=colors[i], label=target_name, alpha=0.7, s=50)

ax.set_xlabel(f'PC1 ({pca_3d.explained_variance_ratio_[0]:.2%} 方差)')
ax.set_ylabel(f'PC2 ({pca_3d.explained_variance_ratio_[1]:.2%} 方差)')
ax.set_zlabel(f'PC3 ({pca_3d.explained_variance_ratio_[2]:.2%} 方差)')
ax.set_title('PCA 3D可视化')
ax.legend()
plt.tight_layout()
plt.show()

print(f"3个主成分累计解释方差: {pca_3d.explained_variance_ratio_.sum():.4f} ({pca_3d.explained_variance_ratio_.sum():.2%})")

## 11. 在高维数据集上的应用（Wine数据集）

In [ ]:
# 加载Wine数据集
wine = load_wine()
X_wine = wine.data
y_wine = wine.target
feature_names_wine = wine.feature_names
target_names_wine = wine.target_names

# 标准化
scaler_wine = StandardScaler()
X_wine_scaled = scaler_wine.fit_transform(X_wine)

print(f"Wine数据集: {X_wine.shape}")
print(f"特征数量: {len(feature_names_wine)}")
print(f"类别数量: {len(target_names_wine)}")

In [ ]:
# PCA降维
pca_wine = PCA()
pca_wine.fit(X_wine_scaled)

# 2D可视化
X_wine_pca = pca_wine.transform(X_wine_scaled)[:, :2]

plt.figure(figsize=(10, 8))
colors = ['red', 'green', 'blue']
for i, target_name in enumerate(target_names_wine):
    mask = (y_wine == i)
    plt.scatter(X_wine_pca[mask, 0], X_wine_pca[mask, 1],
                c=colors[i], label=target_name, alpha=0.7, s=60)

plt.xlabel(f'PC1 ({pca_wine.explained_variance_ratio_[0]:.2%} 方差)')
plt.ylabel(f'PC2 ({pca_wine.explained_variance_ratio_[1]:.2%} 方差)')
plt.title(f'Wine数据集 PCA (累计解释方差: {pca_wine.explained_variance_ratio_[:2].sum():.2%})')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"前2个主成分累计解释方差: {pca_wine.explained_variance_ratio_[:2].sum():.4f} ({pca_wine.explained_variance_ratio_[:2].sum():.2%})")
print(f"前3个主成分累计解释方差: {pca_wine.explained_variance_ratio_[:3].sum():.4f} ({pca_wine.explained_variance_ratio_[:3].sum():.2%})")

In [ ]:
# 累计解释方差曲线
cumulative_variance_wine = np.cumsum(pca_wine.explained_variance_ratio_)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.bar(range(1, len(pca_wine.explained_variance_ratio_) + 1),
        pca_wine.explained_variance_ratio_, alpha=0.7)
plt.xlabel('主成分')
plt.ylabel('解释方差比例')
plt.title('Wine数据集 - 各主成分的解释方差')
plt.grid(True, alpha=0.3, axis='y')

plt.subplot(1, 2, 2)
plt.plot(range(1, len(cumulative_variance_wine) + 1), cumulative_variance_wine,
         'bo-', linewidth=2, markersize=6)
plt.axhline(y=0.95, color='r', linestyle='--', label='95%')
plt.axhline(y=0.90, color='g', linestyle='--', label='90%')
plt.xlabel('主成分数量')
plt.ylabel('累计解释方差比例')
plt.title('Wine数据集 - 累计解释方差')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 找到达到95%解释方差需要的主成分数量
n_components_95 = np.argmax(cumulative_variance_wine >= 0.95) + 1
n_components_90 = np.argmax(cumulative_variance_wine >= 0.90) + 1
print(f"达到90%解释方差需要 {n_components_90} 个主成分")
print(f"达到95%解释方差需要 {n_components_95} 个主成分")

## 12. PCA降维对分类性能的影响

In [ ]:
# 对比使用PCA和不使用PCA的分类性能
X_train_w, X_test_w, y_train_w, y_test_w = train_test_split(
    X_wine_scaled, y_wine, test_size=0.3, random_state=42, stratify=y_wine
)

# 不使用PCA
lr_no_pca = LogisticRegression(random_state=42, max_iter=1000)
lr_no_pca.fit(X_train_w, y_train_w)
score_no_pca = lr_no_pca.score(X_test_w, y_test_w)

# 使用PCA（降维到10个主成分）
pca_lr = PCA(n_components=10)
X_train_w_pca = pca_lr.fit_transform(X_train_w)
X_test_w_pca = pca_lr.transform(X_test_w)

lr_pca = LogisticRegression(random_state=42, max_iter=1000)
lr_pca.fit(X_train_w_pca, y_train_w)
score_pca = lr_pca.score(X_test_w_pca, y_test_w)

print("分类性能对比:")
print(f"不使用PCA: {X_train_w.shape[1]} 个特征, 准确率 = {score_no_pca:.4f}")
print(f"使用PCA: {X_train_w_pca.shape[1]} 个主成分, 准确率 = {score_pca:.4f}")
print(f"特征减少: {X_train_w.shape[1] - X_train_w_pca.shape[1]} ({(1 - X_train_w_pca.shape[1]/X_train_w.shape[1]):.1%})")
print(f"累计解释方差: {pca_lr.explained_variance_ratio_.sum():.2%}")

## 13. PCA vs LDA对比

In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

# PCA降维
pca_compare = PCA(n_components=2)
X_pca_compare = pca_compare.fit_transform(X_wine_scaled)

# LDA降维
lda = LinearDiscriminantAnalysis(n_components=2)
X_lda = lda.fit_transform(X_wine_scaled, y_wine)

# 可视化对比
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# PCA
for i, target_name in enumerate(target_names_wine):
    mask = (y_wine == i)
    axes[0].scatter(X_pca_compare[mask, 0], X_pca_compare[mask, 1],
                  c=colors[i], label=target_name, alpha=0.7, s=60)
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')
axes[0].set_title(f'PCA (无监督)\n累计解释方差: {pca_compare.explained_variance_ratio_.sum():.2%}')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# LDA
for i, target_name in enumerate(target_names_wine):
    mask = (y_wine == i)
    axes[1].scatter(X_lda[mask, 0], X_lda[mask, 1],
                  c=colors[i], label=target_name, alpha=0.7, s=60)
axes[1].set_xlabel('LD1')
axes[1].set_ylabel('LD2')
axes[1].set_title('LDA (有监督)\n解释类间方差')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("PCA vs LDA:")
print("PCA: 无监督降维，最大化数据方差")
print("LDA: 有监督降维，最大化类间分离度")

## 14. 重构误差分析

In [ ]:
# 分析不同主成分数量下的重构误差
n_components_range = range(1, X_wine_scaled.shape[1] + 1)
reconstruction_errors = []

for n_comp in n_components_range:
    pca_recon = PCA(n_components=n_comp)
    X_reduced = pca_recon.fit_transform(X_wine_scaled)
    X_reconstructed = pca_recon.inverse_transform(X_reduced)
    
    # 计算重构误差（MSE）
    error = np.mean((X_wine_scaled - X_reconstructed) ** 2)
    reconstruction_errors.append(error)

# 可视化
plt.figure(figsize=(10, 6))
plt.plot(n_components_range, reconstruction_errors, 'b-o', linewidth=2, markersize=6)
plt.xlabel('主成分数量')
plt.ylabel('重构误差 (MSE)')
plt.title('重构误差 vs 主成分数量')
plt.grid(True, alpha=0.3)
plt.show()

print("重构误差分析:")
for i, (n_comp, error) in enumerate(zip(n_components_range, reconstruction_errors)):
    if i % 5 == 0 or i == len(reconstruction_errors) - 1:
        print(f"  {n_comp}个主成分: {error:.6f}")

## 15. 使用PCA进行噪声过滤

In [ ]:
# 生成带噪声的数据
X_clean, y_noise = make_blobs(
    n_samples=200, n_features=2, centers=3,
    cluster_std=2.0, random_state=42
)

# 添加噪声
noise = np.random.normal(0, 0.5, X_clean.shape)
X_noisy = X_clean + noise

# 标准化
scaler_noise = StandardScaler()
X_noisy_scaled = scaler_noise.fit_transform(X_noisy)

# 使用PCA去噪（保留主要成分）
pca_denoise = PCA(n_components=2)
X_denoised = pca_denoise.fit_transform(X_noisy_scaled)
X_denoised_reconstructed = pca_denoise.inverse_transform(X_denoised)

# 可视化
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].scatter(X_clean[:, 0], X_clean[:, 1], c=y_noise, alpha=0.6, cmap='viridis')
axes[0].set_title('原始数据（无噪声）')
axes[0].set_xlabel('特征 1')
axes[0].set_ylabel('特征 2')
axes[0].grid(True, alpha=0.3)

axes[1].scatter(X_noisy_scaled[:, 0], X_noisy_scaled[:, 1], c=y_noise, alpha=0.6, cmap='viridis')
axes[1].set_title('带噪声数据')
axes[1].set_xlabel('特征 1')
axes[1].set_ylabel('特征 2')
axes[1].grid(True, alpha=0.3)

axes[2].scatter(X_denoised_reconstructed[:, 0], X_denoised_reconstructed[:, 1], c=y_noise, alpha=0.6, cmap='viridis')
axes[2].set_title(f'PCA去噪\n保留{pca_denoise.n_components_}个主成分')
axes[2].set_xlabel('特征 1')
axes[2].set_ylabel('特征 2')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"噪声数据方差: {X_noisy_scaled.var():.4f}")
print(f"去噪后方差: {X_denoised_reconstructed.var():.4f}")
print(f"解释方差比例: {pca_denoise.explained_variance_ratio_.sum():.2%}")

## 总结

### PCA的关键概念

**主成分**: 数据方差最大的方向
**特征向量**: 主成分的方向
**特征值**: 沿主成分方向的方差大小
**解释方差比例**: 特征值占总方差的比例

### PCA的步骤

1. 数据标准化（重要！）
2. 计算协方差矩阵
3. 计算特征值和特征向量
4. 按特征值降序排序
5. 选择前k个主成分
6. 投影数据到主成分空间

### PCA的应用场景

| 应用 | 说明 |
|------|------|
| 降维 | 减少特征数量，降低计算成本 |
| 可视化 | 将高维数据投影到2D/3D |
| 特征提取 | 提取主要变化模式 |
| 去噪 | 保留主要成分，过滤噪声 |
| 数据压缩 | 用更少的维度表示数据 |

### PCA的优缺点

**优点**:
- 简单有效，易于理解
- 无参数调优
- 可以处理高维数据
- 主成分正交，无相关性

**缺点**:
- 假设数据是线性相关的
- 对异常值敏感
- 主成分的可解释性差
- 需要数据标准化
- 只考虑方差，不考虑类别信息

### PCA vs 其他降维方法

| 方法 | 类型 | 特点 |
|------|------|------|
| PCA | 无监督 | 最大化方差 |
| LDA | 有监督 | 最大化类间分离 |
| t-SNE | 无监督 | 保留局部结构 |
| UMAP | 无监督 | 保留全局和局部结构 |

### 实践建议

- 始终先标准化数据
- 使用累计解释方差选择主成分数量（通常保留90%-95%方差）
- 对于分类任务，考虑使用LDA
- 对于可视化，考虑使用t-SNE或UMAP
- 检查特征向量的载荷以理解主成分的含义
- 注意PCA对异常值的敏感性